# 04 — France Beauce Region, Sentinel-2 with GeoAI Engine

Delineates large-field European agriculture in the Beauce region of France using Sentinel-2 imagery and **geoai's** AgricultureFieldDelineator (Mask R-CNN).

**Estimated runtime:** ~15–30 minutes (1 year, GPU).

**Prerequisites:**
```bash
pip install agribound[gee,geoai]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/france_beauce")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "sentinel2"
ENGINE = "geoai"
YEAR = 2023
SAM_REFINE = True

## Create Study Area

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [1.40, 48.10],
                        [1.55, 48.10],
                        [1.55, 48.20],
                        [1.40, 48.20],
                        [1.40, 48.10],
                    ]
                ],
            },
            "properties": {"name": "Beauce AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "beauce_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation

In [ ]:
output_path = OUTPUT_DIR / f"fields_s2_{YEAR}.gpkg"

gdf = agribound.delineate(
    study_area=study_area,
    source=SOURCE,
    year=YEAR,
    engine=ENGINE,
    output_path=str(output_path),
    gee_project=GEE_PROJECT,
    composite_method="median",
    min_area=5000,
    simplify=2.5,
    engine_params={"sam_refine": SAM_REFINE},
)

print(f"Delineated {len(gdf)} fields")
if "metrics:area" in gdf.columns:
    area_ha = gdf["metrics:area"].sum() / 10000
    avg_ha = gdf["metrics:area"].mean() / 10000
    print(f"Total area: {area_ha:,.1f} ha")
    print(f"Average field size: {avg_ha:,.1f} ha")

## Visualization

In [ ]:
m = agribound.show_boundaries(
    gdf,
    basemap="Esri.WorldImagery",
    output_html=str(OUTPUT_DIR / "map_beauce.html"),
)
m